# Module 2: Designing Good Questions

*Getting Started with Jev - Data Trainers LLC - Axel Sirota*

## The story

In Module 1 we built the first triage layer for a personal assistant app: one Jev call per
message returns the team, the urgency and a risk signal, and `triage()` in our code decides what
happens. It works, and it has two loose ends. The risk question joins three conditions with
"or", so when it fires we cannot tell which risk it was. And the team question describes each
team in a few words, which got about 3 in 4 of our 40 messages right.

This module is about writing questions Jev can answer well. Most bad answers are not the model
being wrong: they are a question that asks for several things at once, a state that leaves out
what a person would look at, or options described too vaguely. We fix all three, and finish
with a classifier that finds the exact intent among CLINC150's 150.

## What you will be able to do

- Spot a question that hides several judgments, a date or a sum, and split it into literal ones
- Write instructions and criteria Jev can answer: one judgment each, options described by what they contain
- Design the state: give Jev what a person would look at, in named fields, and name the field in the question
- Compose Jev's answers in code: dates, rules and a two-stage classifier

## Prerequisites

- Module 1 (Choice, Score, Noul, probabilities, confidence; judgments in Jev, decisions in code)
- A Jev API key (given in class). No OpenAI key today: this module is Jev only

## How this notebook works

Three Parts, each in four beats:

| Beat | What happens |
|---|---|
| 1. The problem | what goes wrong, with a real example |
| 2. The solution | the idea that fixes it |
| 3. Demo | we run it together: first the problem, then the solution |
| 4. Lab | you build it, in three levels: Guided, Core, Challenge (optional) |

# Setup

Run the next four cells in order: install, key, config, data. The key cell asks for your Jev key
once; it stays in this runtime only and is never printed.

In [ ]:
%pip install -q typesafe-sdk==0.7.5 datasets==5.1.0

In [ ]:
# ===========================================================================
# KEYS: the Jev key, typed once, never printed
# ===========================================================================
# Why: keys never live in the notebook. If your environment already has it, we use it.
import os
from getpass import getpass

if not os.environ.get("TYPESAFE_API_KEY"):
    os.environ["TYPESAFE_API_KEY"] = getpass("TYPESAFE_API_KEY: ")
if not os.environ["TYPESAFE_API_KEY"].strip():
    raise ValueError("TYPESAFE_API_KEY is empty. Ask your instructor for the class key.")
print("The Jev key is set for this runtime.")

In [ ]:
# ===========================================================================
# CONFIG: client, pinned model, price, a fixed "today", one probe call
# ===========================================================================
# Why: a pinned model id keeps today's numbers reproducible next month, and a fixed TODAY makes
# every date in this notebook resolve the same way on every run.
# The whole room shares one key, and rate limits count per organization. The SDK retries a
# rate-limited call (HTTP 429) on its own, and every loop here calls Jev one message at a time
# on purpose: parallel calls from 25 laptops would hit the limit sooner. If a cell stalls, wait
# a moment and run it again; do not wrap it in your own retry loop.
import statistics
import time
from datetime import date, timedelta

from typesafe_sdk import Choice, Noul, TypeSafeClient

JEV_MODEL = "jev-1.14.0"            # pinned; "jev-latest" moves with each release
JEV_PRICE_IN = 0.042 / 1_000_000    # USD per input token; Jev output is free
TODAY = date(2026, 10, 30)          # a Friday. In production: date.today()

jev = TypeSafeClient()
probe = jev.system_one(state={"message": "hello there"}, model=JEV_MODEL,
                       questions={"greeting": Noul(instructions="Is `message` a greeting?")})
assert probe.model == JEV_MODEL, f"expected {JEV_MODEL}, got {probe.model}"
print(f"Jev answered with {probe.model}: greeting = {probe.answers['greeting'].noul:.2f}")
print(f"TODAY is {TODAY:%A, %B %d, %Y}.")

In [ ]:
# ===========================================================================
# DATA: CLINC150 messages, their intents, and the 10 teams
# ===========================================================================
# Why: today we need the intent of every message, not only its team. The authors' domains.json
# lists the 15 intents of each of the 10 domains (our teams). Out-of-scope messages are left
# out on purpose: Module 3 is about them.
import json
import random
import urllib.request

from datasets import load_dataset

DOMAINS_URL = "https://raw.githubusercontent.com/clinc/oos-eval/master/data/domains.json"
TEAM_INTENTS = json.load(urllib.request.urlopen(DOMAINS_URL, timeout=30))   # team -> its 15 intents
intent_to_team = {intent: team for team, intents in TEAM_INTENTS.items() for intent in intents}

clinc = load_dataset("clinc/clinc_oos", "plus", split="test")
intent_names = clinc.features["intent"].names
in_scope = [(row["text"], intent_names[row["intent"]], intent_to_team[intent_names[row["intent"]]])
            for row in clinc if intent_names[row["intent"]] != "oos"]

TEAMS = {   # Module 1's team descriptions, unchanged
    "banking": "accounts, balances, transfers, bills, direct deposit",
    "credit_cards": "credit cards: limits, rewards, lost or damaged cards, APR",
    "kitchen_and_dining": "recipes, cooking, nutrition, restaurants",
    "home": "smart home, shopping and todo lists, reminders, calendar",
    "auto_and_commute": "car maintenance, gas, traffic, rides, directions",
    "travel": "flights, hotels, luggage, visas, travel alerts, translation",
    "utility": "time, date, weather, timers, calculator, definitions",
    "work": "payroll, PTO, taxes, 401k, meetings",
    "small_talk": "greetings, jokes, chit-chat with the assistant",
    "meta": "settings of the assistant itself: name, language, voice, volume",
}

rng = random.Random(42)   # the same 40 messages as Module 1: 4 per team
sample = []
for team in TEAMS:
    sample += rng.sample([row for row in in_scope if row[2] == team], 4)

print(f"{len(in_scope)} in-scope test messages, {len(intent_to_team)} intents in {len(TEAM_INTENTS)} teams")
print(f"credit_cards owns: {', '.join(TEAM_INTENTS['credit_cards'][:5])}, ...")
for text, intent, team in sample[:3]:
    print(f"  {team:14} {intent:22} | {text}")

## Part 1: One judgment per question, dates in code

Customers who travel abroad ask the credit_cards team for a **travel notice**, so payments from
another country are not blocked. A notice for a trip in the next 7 days is urgent.

### 1. The problem

The quick way: one Noul that says it all, with today's date in the state. The figure shows what
that one question hides, and where it misses.

**Figure: The problem: one Noul hides three judgments and a date check. When it misses, the number cannot say which judgment failed.**

<!-- DIAGRAM: p1-problem-vague-question -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_02/p1-problem-vague-question.png" width="750" alt="Read top to bottom, one band per stage. STATE: three of the 10 messages, cairo on wednesday, oslo on 11/7, boston flight tomorrow, plus today Fri Oct 30. QUESTIONS: one Noul, urgent, hiding three judgments: asks for a notice, when is the trip, within 7 days. JEV ANSWERS: cairo 0.47, oslo 0.49, boston 0.86. YOUR CODE: URGENT if urgent &gt;= 0.7. OUTCOME: cairo MISS, oslo ok by luck, boston MISS although it asked no notice, which judgment failed CANNOT TELL; all 10: vague 7 right, careful wording 6">

### 2. The solution

Same picture; the highlighted boxes changed. Five **literal questions**, each reading one thing
written in the message, and the date maths in **code**.

**The rule: one judgment per question; maths, dates and rules in code.**

**Figure: The solution: literal questions read what is written; code builds the date and applies the rule.**

<!-- DIAGRAM: p1-solution-literal-questions -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_02/p1-solution-literal-questions.png" width="750" alt="Same bands as the problem. STATE: the same three messages and no date, TODAY lives in code. QUESTIONS: five literal ones, asks_notice a Noul, and month, day, weekday and relative_day Choices with a not_stated option. JEV ANSWERS: cairo asks 0.98 wednesday, oslo asks 0.98 november 7, boston asks 0.02 tomorrow. YOUR CODE: trip_date() turns the parts into a date, URGENT if asks &gt;= 0.7 and days &lt;= WINDOW_DAYS. OUTCOME: cairo Nov 4 URGENT, oslo Nov 7 NOT URGENT, boston NOT URGENT, all right, a miss points at one answer; all 10: 10 right, a new policy is 1 constant">

**Figure: Questions don't see each other - one request, five questions, each judged alone against the same state.**

<!-- DIAGRAM: questions-isolated -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_02/questions-isolated.png" width="750" alt="Left to right. STATE, the same for every question: message, oslo on 11/7, travel notice please. ONE REQUEST, each question alone: asks_notice, a Noul, sees message and its criteria; month, a Choice, sees message and its 13 options; day, a Choice, sees message and its 32 options; weekday and relative_day each see message and their own options. NEVER SEEN by a question: its id, the other questions, the other answers. YOUR CODE, where answers meet: trip_date() turns the parts into a date, then URGENT if asks &gt;= 0.7 and days &lt;= WINDOW_DAYS">

So each question names `message` and the trip itself: `day` cannot lean on `month`. The ids are for your code; Jev never receives them (TypeSafe docs, Primitives).

**Figure: Dates in code: trip_date() turns the parts Jev read into a date; one rule decides, and a new policy is one constant.**

<!-- DIAGRAM: p1-dates-in-code -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_02/p1-dates-in-code.png" width="750" alt="Four columns, one per message: cairo, weekday wednesday, becomes Nov 4, days 5; boston, relative_day tomorrow, becomes Oct 31, days 1; oslo, month november and day 7, becomes Nov 7, days 8; london, nothing stated, becomes None. All feed YOUR CODE: URGENT if asks &gt;= 0.7 and days &lt;= WINDOW_DAYS. WINDOW_DAYS 7: cairo URGENT, all 10 right. New policy WINDOW_DAYS 3: cairo NOT URGENT, all 10 right, 0 new Jev calls">

### 3. Demo

Ten travel messages, each with its expected answer; every Noul is yes at 0.7. First the vague
question, then the literal questions with the date in code, then a policy change. Fair warning:
we wrote these 10 and chose 0.7 while building both versions, so the counts illustrate the
failure; they are not a held-out test.

In [ ]:
# ===========================================================================
# DEMO: the problem: one question that hides three judgments and a date (NARRATE)
# ===========================================================================
# Why: the question reads well to a person. Watch where it goes wrong.
TRAVEL_MESSAGES = [   # (message, asks for a notice?, trip date or None if not stated)
    ("i fly to rome on november 5, please set a travel notice on my card", True, date(2026, 11, 5)),
    ("heading to paris tomorrow, can you let my bank know i'm traveling", True, date(2026, 10, 31)),
    ("my trip to cairo is on wednesday, set a travel notice", True, date(2026, 11, 4)),
    ("going to oslo on 11/7, travel notice please", True, date(2026, 11, 7)),
    ("flying to berlin november 9th, please notify my card company", True, date(2026, 11, 9)),
    ("going to london in march, need a travel notice", True, None),
    ("i fly to rome on november 2, what's the weather like there", False, date(2026, 11, 2)),
    ("my flight to boston is tomorrow, is it on time", False, date(2026, 10, 31)),
    ("book a hotel in miami for november 3", False, date(2026, 11, 3)),
    ("i'm visiting madrid this sunday, let my card company know", True, date(2026, 11, 1)),
]
WINDOW_DAYS = 7


def expected_urgent(asks: bool, trip: date | None, window: int) -> bool:
    """The answer we want: the customer asks for a notice AND the trip is within the window."""
    return asks and trip is not None and 0 <= (trip - TODAY).days <= window


vague_question = Noul(instructions="Does `message` need an urgent travel notice, "
                                   "because the trip is within 7 days of `today`?")
careful_question = Noul(instructions="Does the customer in `message` ask for a travel notice "
                                     "for a trip that starts within 7 days of `today`?")
single_tokens = []
right = {"vague": 0, "careful": 0}
print(f"{'message':52} {'expect':>6} {'vague':>9} {'careful':>9}")
for text, asks, trip in TRAVEL_MESSAGES:
    r = jev.system_one(state={"message": text, "today": f"{TODAY:%A, %B %d, %Y}"}, model=JEV_MODEL,
                       questions={"vague": vague_question, "careful": careful_question})
    single_tokens.append(r.usage.input_tokens)   # two questions in this call, read together
    expect = expected_urgent(asks, trip, WINDOW_DAYS)
    marks = {}
    for name in right:
        ok = (r.answers[name].noul >= 0.7) == expect   # yes at 0.7, as in the literal rule
        right[name] += ok
        marks[name] = f"{r.answers[name].noul:.2f} {'ok' if ok else 'MISS'}"
    print(f"{text[:52]:52} {str(expect):>6} {marks['vague']:>9} {marks['careful']:>9}")
print(f"\nRight at the 0.7 line: vague {right['vague']} of 10, careful wording {right['careful']} of 10.")
print(f"Input tokens per call with both questions: {statistics.mean(single_tokens):.0f}")
print("Better words did not fix it. When it misses, which of the three judgments failed? The number cannot say.")

In [ ]:
# ===========================================================================
# HELPER: the literal questions, and trip_date() in code (NARRATE)
# ===========================================================================
# Why: each question reads one thing that is written in the message. The `not_stated` options
# matter: a Choice always picks one of its options, so "no month" must be one of them.
# An option written as {name: None} has no description: its name says it all ("march", "12").
MONTHS = ["january", "february", "march", "april", "may", "june", "july", "august",
          "september", "october", "november", "december"]
WEEKDAYS = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]

LITERAL_QUESTIONS = {
    "asks_notice": Noul(
        instructions="Does `message` ask to tell the bank or card company about upcoming travel?",
        criteria={"true": "the customer asks for a travel notice, travel alert or travel notification",
                  "false": "the customer mentions a trip but asks for something else, such as "
                           "the weather, a flight status or a hotel"}),
    "month": Choice(instructions="In which month is the trip in `message`?",
                    criteria={**{m: None for m in MONTHS}, "not_stated": "the message names no month"}),
    "day": Choice(instructions="On which day of the month is the trip in `message`?",
                  criteria={**{str(d): None for d in range(1, 32)},
                            "not_stated": "the message names no day of the month"}),
    "weekday": Choice(instructions="On which day of the week is the trip in `message`?",
                      criteria={**{w: None for w in WEEKDAYS},
                                "not_stated": "the message names no day of the week"}),
    "relative_day": Choice(instructions="Does `message` say the trip is today or tomorrow?",
                           criteria={"today": "the trip is today", "tomorrow": "the trip is tomorrow",
                                     "not_stated": "the message says neither today nor tomorrow"}),
}


def trip_date(answers: dict, today: date = TODAY) -> date | None:
    """Turn the date parts Jev read into a calendar date. None when no date is stated."""
    if answers["relative_day"].choice == "today":
        return today
    if answers["relative_day"].choice == "tomorrow":
        return today + timedelta(days=1)
    month, day = answers["month"].choice, answers["day"].choice
    if month != "not_stated" and day != "not_stated":
        try:
            trip = date(today.year, MONTHS.index(month) + 1, int(day))
        except ValueError:                     # february 30: code catches it, Jev never could
            return None
        # a date already past means next year: fine for trips, wrong for "last tuesday"
        return trip if trip >= today else trip.replace(year=today.year + 1)
    if answers["weekday"].choice != "not_stated":   # the next such weekday, today included
        return today + timedelta(days=(WEEKDAYS.index(answers["weekday"].choice) - today.weekday()) % 7)
    return None


print(f"{len(LITERAL_QUESTIONS)} literal questions ready; trip_date() turns their answers into a date.")

In [ ]:
# ===========================================================================
# DEMO: the solution: literal questions, the date and the rule in code (NARRATE)
# ===========================================================================
# Why: same 10 messages. Jev reads what is written; trip_date() and one comparison decide.
literal_answers = {}
literal_tokens = []
right = 0
print(f"{'message':52} {'expect':>6} {'asks':>5} {'trip':>11} {'days':>5}")
for text, asks, trip in TRAVEL_MESSAGES:
    r = jev.system_one(state={"message": text}, model=JEV_MODEL, questions=LITERAL_QUESTIONS)
    literal_answers[text] = r.answers
    literal_tokens.append(r.usage.input_tokens)
    asks_p = r.answers["asks_notice"].noul
    read_trip = trip_date(r.answers)
    days = (read_trip - TODAY).days if read_trip else None
    urgent = asks_p >= 0.7 and days is not None and 0 <= days <= WINDOW_DAYS   # the rule, in code
    expect = expected_urgent(asks, trip, WINDOW_DAYS)
    right += urgent == expect
    print(f"{text[:52]:52} {str(expect):>6} {asks_p:>5.2f} {str(read_trip):>11} {str(days):>5}  "
          f"{'ok' if urgent == expect else 'MISS'}")
print(f"\nLiteral questions + code: {right} of {len(TRAVEL_MESSAGES)} right.")
print(f"Input tokens per message: {statistics.mean(literal_tokens):.0f} "
      f"(${1000 * statistics.mean(literal_tokens) * JEV_PRICE_IN:.3f} per 1,000 messages)")

In [ ]:
# ===========================================================================
# DEMO: the policy changes to 3 days: one constant, zero Jev calls (NARRATE)
# ===========================================================================
# Why: the window lives in code, so a new policy reuses the answers we already have.
new_window = 3   # the new policy; WINDOW_DAYS stays 7 for the rest of the notebook
right = 0
for text, asks, trip in TRAVEL_MESSAGES:
    answers = literal_answers[text]
    read_trip = trip_date(answers)
    urgent = (answers["asks_notice"].noul >= 0.7 and read_trip is not None
              and 0 <= (read_trip - TODAY).days <= new_window)
    right += urgent == expected_urgent(asks, trip, new_window)
    if urgent:
        print(f"  urgent within {new_window} days: {text}")
print(f"\nNew policy: {right} of {len(TRAVEL_MESSAGES)} right, with 0 new Jev calls.")
print("With the vague question you would reword it, call Jev again, and hope.")

**Figure: Part 1 results: how many of the 10 messages each version gets right.**

<!-- DIAGRAM: p1-demo-results -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_02/p1-demo-results.png" width="700" alt="Bar chart of right answers out of 10 travel messages at the 0.7 line: vague question 7, careful wording 6, literal questions plus code 10, and the 3-day rule, with 0 new Jev calls, 10">

**Figure: Part 1 cost: input tokens per message. The literal questions cost more; the demo's single-question line sends the vague and careful Nouls together.**

<!-- DIAGRAM: p1-demo-tokens -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_02/p1-demo-tokens.png" width="700" alt="Bar chart of input tokens per message: the two single Nouls in one call 360, the five literal questions 932">

> **Think About It**: The literal version costs more input tokens per message than the vague
> call: compare the two token lines the demo printed. What did those tokens buy you? Point at
> one MISS in the vague table and say which of the three hidden judgments caused it. Could you
> have done that before the split?

### 4. Lab 1: Split the risk question

Module 1 left us a risk question that joins three conditions with "or": *"Does `message` report
a lost or stolen card, a charge the customer did not make, or a locked account?"* When it fires
we do not know which risk it was, and the "or" blurs the cases in between. You split it.

#### Your Task

1. Write three literal risk questions, one per condition (Level 1)
2. Write `assess_risk()`: one Jev call, then a truth table in code (a fixed list of rules that
   maps the flags to an action) that returns an action and the reasons (Level 2)
3. Optional: teach `trip_date()` to read "next friday" (Level 3)

#### Level 1: Guided

Fill the three instructions: `lost_or_stolen_instructions`, `unknown_charge_instructions` and
`locked_out_instructions`. Each is one yes/no statement about `message`, with one condition
only. The cell runs your three questions and Module 1's "or" question on 8 messages, side by
side. Stuck? In the exercises notebook, the FALLBACK cell below fills them in so you can move on.

#### Level 2: Core

Write the body of `assess_risk(message)`: one `jev.system_one` call with `RISK_QUESTIONS`, then
the truth table in the Lab 1 map below, checked in order: any risk >= 0.7 is ESCALATE (reasons:
those risks), else any risk above 0.3 is REVIEW (reasons: those risks), else NONE (no reasons).

#### Level 3: Challenge (optional)

`trip_date()` reads "next friday" as today, because it only sees the weekday. Add a `week`
Choice and write `trip_date_next(answers)`: "next friday" is the Friday of next week (weeks start
on Monday). Nothing later uses it.

#### Expected Output

- Level 1: the "or" question gives one number per message; your three show which risk it is.
  The app login scores about 0.5 on the "or" question but near 1.0 on `locked_out`; "someone
  used my card but i'm not sure" lands near 0.5 on `unknown_charge`
- Level 2: an 8-row table with 4 ESCALATE (lost card, unknown charge, app login, and the stolen
  card with two reasons), 1 REVIEW (someone used my card), 3 NONE
- Level 3: four messages with the date you read and the expected one; all four match

#### Hints

- Level 1: say the condition the way a person would point at it, about `message`, so that
  "true" means risk. For a shop it might read "Does `message` report that the package arrived
  damaged?"
- Level 2: `response.answers[name].noul` for each name in `RISK_QUESTIONS`; build the
  `reasons` lists with comprehensions
- Level 3: `TODAY - timedelta(days=TODAY.weekday())` is this week's Monday

**Figure: Lab 1 map: what you build at each level. Level 2's truth table is the one to code.**

<!-- DIAGRAM: p1-lab-map -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_02/p1-lab-map.png" width="750" alt="The message, card stolen and charges i don&#x27;t recognize, goes to Level 1&#x27;s three literal Nouls, lost_or_stolen, unknown_charge and locked_out; against Module 1&#x27;s or question the app login scores 0.50 on or and 0.97 on locked_out. Level 2 assess_risk makes 1 Jev call, then any flag &gt;= 0.7 is ESCALATE with those flags as reasons, else any &gt; 0.3 is REVIEW with the flags between 0.3 and 0.7, else NONE with no reasons; it returns flags, action and reasons: this message ESCALATE with 2 reasons, 8 messages give 4 ESCALATE, 1 REVIEW, 3 NONE. Below, dashed, Level 3 optional: trip_date() plus a week Choice gives trip_date_next(), next friday Nov 6">

In [ ]:
# ===========================================================================
# SOLUTION: LAB 1.1 (Guided)
# ===========================================================================
# Approach:
# - One condition per question, said the way a person would point at it, each about `message`.
# - Lost and stolen stay together: both mean the card is out of the customer's hands, and the
#   action is the same. Split them only if the rules ever treat them differently.
# Step 1: one literal yes/no statement per risk, each about `message`, one condition each.
lost_or_stolen_instructions = "Does `message` report that the customer's card is lost or stolen?"
unknown_charge_instructions = "Does `message` report a charge the customer did not make?"
locked_out_instructions = "Does `message` report that the customer cannot get into their account?"

OR_QUESTION = Noul(instructions="Does `message` report a lost or stolen card, a charge the "
                                "customer did not make, or a locked account?")   # Module 1's
LAB1_MESSAGES = [
    "i lost my card at the mall and need it frozen right away",
    "there's a charge on my statement i never made",
    "i think someone used my card but i'm not sure",
    "i can't find my wallet, maybe it's in the car",
    "the bank app won't let me log in",
    "why was my card declined at the gas station",
    "my card was stolen and there are charges i don't recognize",
    "what's my checking balance",
]

RISK_QUESTIONS = {
    "lost_or_stolen": Noul(instructions=lost_or_stolen_instructions),
    "unknown_charge": Noul(instructions=unknown_charge_instructions),
    "locked_out": Noul(instructions=locked_out_instructions),
}
print(f"{'message':52} {'or':>5} | {'lost':>5} {'charge':>6} {'locked':>6}")
for text in LAB1_MESSAGES:
    a = jev.system_one(state={"message": text}, model=JEV_MODEL,
                       questions={"or": OR_QUESTION, **RISK_QUESTIONS}).answers
    print(f"{text[:52]:52} {a['or'].noul:>5.2f} | {a['lost_or_stolen'].noul:>5.2f} "
          f"{a['unknown_charge'].noul:>6.2f} {a['locked_out'].noul:>6.2f}")
# Explanation:
# - The "or" question gives the app login about 0.5: unsettled. Asked on its own, locked_out
#   is near 1.0. Joining conditions blurs each one, and you never learn which fired.
# - "my card was stolen and there are charges i don't recognize" lights up two flags: with
#   separate questions, more than one can be true; a single "or" cannot tell you that.
# - Common mistake: "Is the customer's card safe?" That is a negation of what you want to know;
#   ask the condition directly so that "true" means risk.

In [ ]:
# ===========================================================================
# SOLUTION: LAB 1.2 (Core)
# ===========================================================================
# Approach:
# - One call for the three questions (they share the state), then a truth table in code. The
#   middle band (0.3 to 0.7) is its own row: Jev is telling us the message does not settle it.
def assess_risk(message: str) -> dict:
    """One Jev call with RISK_QUESTIONS, then the truth table in code.

    Returns {"flags": {name: noul}, "action": str, "reasons": list[str]}:
    any flag >= 0.7 -> "ESCALATE", reasons = those flags;
    else any flag > 0.3 -> "REVIEW", reasons = those flags;
    else "NONE", reasons = [].
    """
    response = jev.system_one(state={"message": message}, model=JEV_MODEL, questions=RISK_QUESTIONS)
    flags = {name: response.answers[name].noul for name in RISK_QUESTIONS}
    high = [name for name, p in flags.items() if p >= 0.7]
    unsettled = [name for name, p in flags.items() if 0.3 < p < 0.7]
    if high:
        return {"flags": flags, "action": "ESCALATE", "reasons": high}
    if unsettled:
        return {"flags": flags, "action": "REVIEW", "reasons": unsettled}
    return {"flags": flags, "action": "NONE", "reasons": []}


print(f"{'message':52} {'action':9} reasons")
for text in LAB1_MESSAGES:
    result = assess_risk(text)
    print(f"{text[:52]:52} {result['action']:9} {', '.join(result['reasons'])}")
# Explanation:
# - Every ESCALATE now says WHY. A person picking it up knows whether to freeze a card, dispute
#   a charge or get the customer back into the account.
# - "someone used my card but i'm not sure" lands in the middle band on unknown_charge: REVIEW.
#   Module 1's "or" question pushed it to about 0.8 and escalated it.
# - "i can't find my wallet, maybe it's in the car" is NONE now: the customer does not report a
#   lost card yet. If your policy wants a person on that, add a rule; do not reword the question.

In [ ]:
# ===========================================================================
# SOLUTION: LAB 1.3 (Challenge, optional)
# ===========================================================================
# Approach:
# - Jev reads one more thing that is written ("next" or "this" week); the calendar maths stays
#   in code, built on this week's Monday.
# Spec: next_questions() returns LITERAL_QUESTIONS plus a "week" Choice with the options
# "this_week", "next_week" and "not_stated". trip_date_next(answers) works like trip_date()
# but reads "next <weekday>" as that weekday in next week (weeks start on Monday).
# The test prints, per message: what trip_date() read, what yours reads, the expected date.
NEXT_WEEK_MESSAGES = [
    ("my trip to lisbon starts next friday, set a travel notice", date(2026, 11, 6)),
    ("i'm visiting madrid this sunday, let my card company know", date(2026, 11, 1)),
    ("i leave for canada on monday, add a travel notification", date(2026, 11, 2)),
    ("flying to denver next wednesday, travel notice please", date(2026, 11, 4)),
]


def next_questions() -> dict:
    week = Choice(instructions="Which week is the trip in `message`?", criteria={
        "this_week": "the message says this week, such as this sunday",
        "next_week": "the message says next week, such as next friday",
        "not_stated": "the message names a day but not the week, or no day at all"})
    return {**LITERAL_QUESTIONS, "week": week}


def trip_date_next(answers: dict, today: date = TODAY) -> date | None:
    weekday, week = answers["weekday"].choice, answers["week"].choice
    if weekday != "not_stated" and week != "not_stated" and answers["relative_day"].choice == "not_stated":
        monday = today - timedelta(days=today.weekday())
        return monday + timedelta(days=WEEKDAYS.index(weekday) + (7 if week == "next_week" else 0))
    return trip_date(answers, today)


questions = next_questions()
for text, expected in NEXT_WEEK_MESSAGES:
    answers = jev.system_one(state={"message": text}, model=JEV_MODEL, questions=questions).answers
    print(f"{text[:50]:50} trip_date {str(trip_date(answers)):>10}   "
          f"yours {str(trip_date_next(answers)):>10}   expected {expected}")
# Explanation:
# - trip_date() reads "next friday" as today (Friday): the weekday alone is not enough.
# - "Next friday" is ambiguous even for people; the convention (weeks start on Monday) lives in
#   code, where you can read it and change it, not inside a question.

## Part 2: Give Jev what a person would look at

Our risk questions are literal now. But they read one message, and customers write a
conversation.

### 1. The problem

Most pipelines send Jev the latest line of the chat. The questions were fine; **the state left
out what a person would look at**.

**Figure: The problem: the state holds only the last line, so the lost card never reaches Jev and the rules say NONE with no error.**

<!-- DIAGRAM: p2-problem-last-line -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_02/p2-problem-last-line.png" width="420" alt="Read top to bottom. THE CHAT: i can&#x27;t find my card anywhere, then ok so what do i do now. STATE: message holds the last line only; the lost card line is LEFT OUT. QUESTIONS: three Nouls about message. JEV ANSWERS: lost_or_stolen 0.02, unknown_charge 0.01, locked_out 0.04. YOUR CODE risk_decision(): &gt;= 0.7 ESCALATE, &gt; 0.3 REVIEW, else NONE. OUTCOME: NONE, the lost card MISSED, no error raised; over 6 threads only 1 ESCALATE">

### 2. The solution

Same picture; the highlighted boxes changed. The state is the **whole thread** as one named
field, every turn with who said it, and each question **names that field**.

**The rule: the state is the briefing; a question can only judge what is in it, by name.**

**Figure: The solution: the whole thread goes into the state as `thread`, the questions name it, and the same code escalates.**

<!-- DIAGRAM: p2-solution-thread-state -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_02/p2-solution-thread-state.png" width="510" alt="Same bands as the problem. STATE: thread, a list of turns, each with from customer and its text. QUESTIONS: three Nouls that name thread. JEV ANSWERS: lost_or_stolen 0.77, unknown_charge 0.02, locked_out 0.05. YOUR CODE: the same risk_decision(). OUTCOME: ESCALATE, reason lost_or_stolen; over 6 threads 3 to 4 ESCALATE and the found card REVIEW; the stolen wallet scores 0.66 to 0.71, at the line, so ESCALATE or REVIEW">

### 3. Demo

Six short chat threads, four state designs (the figure below), the same rules every time. The
first cell turns Lab 1's rules into a function.

**Figure: The four state designs the demo runs, in cell order, scored on the lost card thread: 1 is the problem, 2 the solution, 3 and 4 the contrast.**

<!-- DIAGRAM: p2-demo-state-designs -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_02/p2-demo-state-designs.png" width="530" alt="Four panels, each STATE then QUESTIONS then lost_or_stolen for the lost card thread: 1 last line, message holds the last line, questions about message, 0.02 NONE; 2 whole thread, thread with every turn, questions that name thread, 0.77 ESCALATE; 3 lazy fix, earlier_messages plus message, questions about message, 0.48 REVIEW; 4 control, thread with every turn but questions about message, 0.70, at the line, varies by run">

In [ ]:
# ===========================================================================
# RECAP: the Lab 1 truth table as a function, and the literal risk questions
# ===========================================================================
# Why: the demo needs fixed questions and fixed rules, whatever you wrote in Lab 1.
MESSAGE_RISK_QUESTIONS = {
    "lost_or_stolen": Noul(instructions="Does `message` report that the customer's card is lost or stolen?"),
    "unknown_charge": Noul(instructions="Does `message` report a charge the customer did not make?"),
    "locked_out": Noul(instructions="Does `message` report that the customer cannot get into their account?"),
}


def risk_decision(flags: dict) -> dict:
    """Lab 1's truth table: ESCALATE on any flag >= 0.7, REVIEW on any above 0.3, else NONE."""
    high = [name for name, p in flags.items() if p >= 0.7]
    unsettled = [name for name, p in flags.items() if 0.3 < p < 0.7]
    if high:
        return {"action": "ESCALATE", "reasons": high}
    if unsettled:
        return {"action": "REVIEW", "reasons": unsettled}
    return {"action": "NONE", "reasons": []}


print("risk_decision() ready:", risk_decision({"lost_or_stolen": 0.9, "unknown_charge": 0.1}))

In [ ]:
# ===========================================================================
# DEMO: the problem: Jev sees only the last line (NARRATE)
# ===========================================================================
# Why: this is what most pipelines send. The questions are fine; the state is not.
THREADS = [
    ["i can't find my card anywhere", "ok so what do i do now"],
    ["there's a $300 charge from a store i've never been to", "can you help me with that"],
    ["the app says my password is wrong", "and now it says too many attempts, i'm stuck"],
    ["i lost my card yesterday", "never mind, found it in my jacket"],
    ["my wallet was stolen on the train", "please hurry"],
    ["what's my checking balance", "thanks"],
]


def flags_of(answers) -> dict:
    return {name: answers[name].noul for name in MESSAGE_RISK_QUESTIONS}


last_line = []
print(f"{'last line only':45} {'lost':>5} {'charge':>6} {'locked':>6}  action")
for thread in THREADS:
    r = jev.system_one(state={"message": thread[-1]}, model=JEV_MODEL, questions=MESSAGE_RISK_QUESTIONS)
    flags = flags_of(r.answers)
    last_line.append(risk_decision(flags)["action"])
    print(f"{thread[-1][:45]:45} {flags['lost_or_stolen']:>5.2f} {flags['unknown_charge']:>6.2f} "
          f"{flags['locked_out']:>6.2f}  {last_line[-1]}")

In [ ]:
# ===========================================================================
# DEMO: the solution: the whole thread, named, and questions that name it (NARRATE)
# ===========================================================================
# Why: same threads, same rules. Only the state and the field the questions point at change.
THREAD_RISK_QUESTIONS = {
    "lost_or_stolen": Noul(instructions="Does the customer in `thread` report that their card is lost or stolen?"),
    "unknown_charge": Noul(instructions="Does the customer in `thread` report a charge they did not make?"),
    "locked_out": Noul(instructions="Does the customer in `thread` report that they cannot get into their account?"),
}


def thread_state(lines: list[str]) -> dict:
    return {"thread": [{"from": "customer", "text": line} for line in lines]}


print(json.dumps(thread_state(THREADS[0]), indent=2))
whole_thread = []
print(f"\n{'whole thread':45} {'lost':>5} {'charge':>6} {'locked':>6}  action   (last line only)")
for thread, before in zip(THREADS, last_line):
    r = jev.system_one(state=thread_state(thread), model=JEV_MODEL, questions=THREAD_RISK_QUESTIONS)
    flags = flags_of(r.answers)
    whole_thread.append(risk_decision(flags)["action"])
    print(f"{' / '.join(thread)[:45]:45} {flags['lost_or_stolen']:>5.2f} {flags['unknown_charge']:>6.2f} "
          f"{flags['locked_out']:>6.2f}  {whole_thread[-1]:8} ({before})")

In [ ]:
# ===========================================================================
# DEMO: a contrast, the lazy fix: add the history, keep asking about `message` (NARRATE)
# ===========================================================================
# Why: the lazy fix changes two things at once: the earlier lines go into a new field, and the
# questions still name `message`, which now holds the last line only. The second loop changes
# one thing: the same `thread` state as the solution, with the `message` questions.
lazy_fix, thread_message_q = [], []
print(f"{'lazy fix: earlier_messages + message':45} {'lost':>5} {'charge':>6} {'locked':>6}  action")
for thread in THREADS:
    r = jev.system_one(state={"earlier_messages": thread[:-1], "message": thread[-1]},
                       model=JEV_MODEL, questions=MESSAGE_RISK_QUESTIONS)
    flags = flags_of(r.answers)
    lazy_fix.append(risk_decision(flags)["action"])
    print(f"{' / '.join(thread)[:45]:45} {flags['lost_or_stolen']:>5.2f} {flags['unknown_charge']:>6.2f} "
          f"{flags['locked_out']:>6.2f}  {lazy_fix[-1]}")

print(f"\n{'thread state, questions name `message`':45} {'lost':>5} {'charge':>6} {'locked':>6}  action")
for thread in THREADS:
    r = jev.system_one(state=thread_state(thread), model=JEV_MODEL, questions=MESSAGE_RISK_QUESTIONS)
    flags = flags_of(r.answers)
    thread_message_q.append(risk_decision(flags)["action"])
    print(f"{' / '.join(thread)[:45]:45} {flags['lost_or_stolen']:>5.2f} {flags['unknown_charge']:>6.2f} "
          f"{flags['locked_out']:>6.2f}  {thread_message_q[-1]}")

print("\nESCALATE, of 6 threads: last line", last_line.count("ESCALATE"),
      "| lazy fix", lazy_fix.count("ESCALATE"),
      "| thread, `message` questions", thread_message_q.count("ESCALATE"),
      "| thread, `thread` questions", whole_thread.count("ESCALATE"))
print("A question reads the field it names. Name the last line, and the earlier lines are only half read.")


**Figure: Part 2 results, all 6 threads: how many escalate under each design, in cell order.**

<!-- DIAGRAM: p2-demo-results -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_02/p2-demo-results.png" width="700" alt="Bar chart of threads escalated out of 6: design 1 last line 1, design 2 whole thread 3, design 3 lazy fix 1, design 4 control 3">

**Figure: Part 2 results: lost_or_stolen for the lost card thread under each design, against the 0.7 line.**

<!-- DIAGRAM: p2-demo-lost-card -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_02/p2-demo-lost-card.png" width="700" alt="Bar chart with a line at 0.7: design 1 last line 0.02, design 2 whole thread 0.77, design 3 lazy fix 0.48, design 4 control 0.70">

> **Think About It**: Look at "i lost my card yesterday / never mind, found it in my jacket" in the
> thread table. `lost_or_stolen` lands in the middle, so the rules send it to REVIEW. Is Jev
> wrong? The card WAS lost, and now it is not. Which judgment is missing from our questions, and
> where would its rule go? Level 3 of the lab answers that.

### 4. Lab 2: Build the state from a support ticket

Real chats arrive as tickets from the support system: ids, a channel, timestamps, and turns
from both the customer and our agent. You decide what Jev reads, then assess every ticket.

#### Your Task

1. Write the state for one ticket: what goes in, what stays out (Level 1)
2. Write `assess_thread()`: ticket in, state built, one Jev call, `risk_decision()` out (Level 2)
3. Optional: add the missing judgment, "the problem is already solved" (Level 3)

#### Level 1: Guided

Fill `ticket_state`: the state Jev should read for `TICKET`. The questions in
`THREAD_RISK_QUESTIONS` read a field called `thread`, so your state must have one. Decide what
else goes in (ids? the channel? our agent's lines?) and how each turn shows who said it. The
cell prints your state and the three risk flags for it.

#### Level 2: Core

Write the body of `assess_thread(ticket)`: build the state from `ticket["turns"]` the way you
decided in Level 1, make one call with `THREAD_RISK_QUESTIONS`, and return the flags plus what
`risk_decision()` says. The cell prints a row per ticket.

#### Level 3: Challenge (optional)

Write `assess_thread_resolved(ticket)`: the same call plus a fourth Noul, `resolved`, "the
customer says the problem is already solved". Add a rule, checked first: resolved >= 0.7 ->
action CLOSE. Print the six tickets again. Nothing later uses it.

#### Expected Output

- Level 1: `unknown_charge` high, `lost_or_stolen` low (the customer says the card is right
  there). On our runs, keeping the ids or dropping who said each line changed no decision on
  these tickets: leaving them out is hygiene, and the solution explains why we still do it
- Level 2: six rows: ESCALATE for the lost card, the unknown charge and the locked account;
  the found card goes to REVIEW; the balance question is NONE. The stolen wallet scored 0.66 to
  0.71 on our runs, right at the 0.7 line, so ESCALATE and REVIEW both count as correct there
- Level 3: the found card becomes CLOSE; the other five rows keep their action

#### Hints

- Level 1: Jev does not know who "C-2210" is, and a ticket id is not evidence. Who said a line
  matters: our agent asking "is the card still with you?" is not the customer reporting a lost card
- Level 2: `ticket["turns"]` is a list of `(who, text)` pairs; build the same shape as Level 1
- Level 3: the new question names the field too: "Does the customer in `thread` say ...?"

**Figure: Lab 2 map: what you build, and how the three levels connect.**

<!-- DIAGRAM: p2-lab-map -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_02/p2-lab-map.png" width="750" alt="TICKET T-88202: ticket_id, customer_id, channel chat; customer: a $300 charge from a store i&#x27;ve never been to; agent: is the card still with you?; customer: yes, it&#x27;s right here. Level 1, ticket_state, must hold thread because the questions name it; you decide ids, agent lines and how each turn shows from and text; it should give unknown_charge 0.98 and lost_or_stolen 0.05. Level 2 assess_thread builds that state, makes 1 Jev call with THREAD_RISK_QUESTIONS and applies the given risk_decision; over 6 tickets it returns 3 ESCALATE (lost, charge, locked), the found card REVIEW, the stolen wallet 0.66 to 0.71 ESCALATE or REVIEW, and balance NONE. Below, dashed, Level 3 optional: a resolved Noul, checked first, resolved &gt;= 0.7 gives CLOSE for the found card">

In [ ]:
# ===========================================================================
# SOLUTION: LAB 2.1 (Guided)
# ===========================================================================
# Approach:
# - In: every turn, in order, with who said it. Out: ticket id, customer id and channel, which
#   no person would use to judge the risk.
TICKETS = [
    {"ticket_id": "T-88201", "customer_id": "C-1042", "channel": "chat", "turns": [
        ("customer", "hi, quick question about my card"),
        ("agent", "sure, what's going on?"),
        ("customer", "i can't find it anywhere, i think i left it on the bus")]},
    {"ticket_id": "T-88202", "customer_id": "C-2210", "channel": "chat", "turns": [
        ("customer", "there's a $300 charge from a store i've never been to"),
        ("agent", "i can help with that. is the card still with you?"),
        ("customer", "yes, it's right here")]},
    {"ticket_id": "T-88203", "customer_id": "C-0317", "channel": "chat", "turns": [
        ("customer", "the app says my password is wrong"),
        ("agent", "please try resetting it from the login screen"),
        ("customer", "did that, now it says too many attempts")]},
    {"ticket_id": "T-88204", "customer_id": "C-5560", "channel": "chat", "turns": [
        ("customer", "i lost my card yesterday"),
        ("agent", "i'm sorry to hear that. shall i freeze it?"),
        ("customer", "no need, found it in my jacket")]},
    {"ticket_id": "T-88205", "customer_id": "C-7781", "channel": "chat", "turns": [
        ("customer", "my wallet was stolen on the train"),
        ("agent", "i'm sorry. let's get your cards frozen"),
        ("customer", "please hurry")]},
    {"ticket_id": "T-88206", "customer_id": "C-0904", "channel": "chat", "turns": [
        ("customer", "can you tell me my checking balance"),
        ("agent", "it's $1,240"),
        ("customer", "thanks!")]},
]
TICKET = TICKETS[1]

# Step 1: the state Jev should read for TICKET. Which fields go in? How does a turn say who spoke?
ticket_state = {"thread": [{"from": who, "text": text} for who, text in TICKET["turns"]]}

print(json.dumps(ticket_state, indent=2))
a = jev.system_one(state=ticket_state, model=JEV_MODEL, questions=THREAD_RISK_QUESTIONS).answers
print({name: round(a[name].noul, 2) for name in THREAD_RISK_QUESTIONS})
# Explanation:
# - The field is called "thread" because THREAD_RISK_QUESTIONS name `thread`.
# - "from" keeps our agent's question ("is the card still with you?") from reading as the
#   customer's report. The customer's answer ("it's right here") keeps lost_or_stolen low.
# - Ids are for code: keep them in your log next to the answer, not in the state. On these
#   tickets, ids in or "from" dropped changed no decision (measured); they cost tokens and, as
#   the docs warn, an unneeded field can sway a question you add later.

In [ ]:
# ===========================================================================
# SOLUTION: LAB 2.2 (Core)
# ===========================================================================
# Approach:
# - The Level 1 decision becomes a function; the rules are the same risk_decision() as Lab 1.
def assess_thread(ticket: dict) -> dict:
    """Build the state from ticket["turns"], one Jev call with THREAD_RISK_QUESTIONS.

    Returns {"flags": {name: noul}, "action": str, "reasons": list[str]}, where action and
    reasons come from risk_decision(flags).
    """
    state = {"thread": [{"from": who, "text": text} for who, text in ticket["turns"]]}
    response = jev.system_one(state=state, model=JEV_MODEL, questions=THREAD_RISK_QUESTIONS)
    flags = {name: response.answers[name].noul for name in THREAD_RISK_QUESTIONS}
    return {"flags": flags, **risk_decision(flags)}


print(f"{'ticket':8} {'last customer line':42} {'action':9} reasons")
for ticket in TICKETS:
    result = assess_thread(ticket)
    print(f"{ticket['ticket_id']:8} {ticket['turns'][-1][1][:42]:42} {result['action']:9} "
          f"{', '.join(result['reasons'])}")
# Explanation:
# - Read the last lines: "please hurry", "thanks!", "did that, now it says too many attempts".
#   Most carry no risk on their own; the thread does.
# - The found card goes to REVIEW: the card WAS lost, so lost_or_stolen sits in the middle.
#   That is a missing judgment, not a wrong one (Level 3).
# - The stolen wallet scores near 0.7: it reports a stolen wallet, and the card is only implied.
#   If that row matters to you, the fix is in the state or a question, not a lower threshold.

In [ ]:
# ===========================================================================
# SOLUTION: LAB 2.3 (Challenge, optional)
# ===========================================================================
# Approach:
# - A fourth literal question in the same call, and one new row at the top of the truth table.
# Spec: assess_thread_resolved(ticket) asks THREAD_RISK_QUESTIONS plus a fourth Noul,
# "resolved" (the customer says the problem is already solved), in ONE call. Rules, in order:
# resolved >= 0.7 -> "CLOSE"; otherwise risk_decision() on the three risk flags.
RESOLVED_QUESTION = Noul(instructions="Does the customer in `thread` say the problem is already solved?")


def assess_thread_resolved(ticket: dict) -> dict:
    state = {"thread": [{"from": who, "text": text} for who, text in ticket["turns"]]}
    response = jev.system_one(state=state, model=JEV_MODEL,
                              questions={**THREAD_RISK_QUESTIONS, "resolved": RESOLVED_QUESTION})
    flags = {name: response.answers[name].noul for name in THREAD_RISK_QUESTIONS}
    resolved = response.answers["resolved"].noul
    if resolved >= 0.7:
        return {"flags": flags, "resolved": resolved, "action": "CLOSE", "reasons": ["resolved"]}
    return {"flags": flags, "resolved": resolved, **risk_decision(flags)}


for ticket in TICKETS:
    result = assess_thread_resolved(ticket)
    print(f"{ticket['ticket_id']:8} {result['action']:9} resolved {result['resolved']:.2f}")
# Explanation:
# - We did not reword lost_or_stolen to "lost and not found again": that would be two
#   judgments in one question. "Resolved" is its own judgment, and the order of the rules
#   is a business decision in code.
# - The balance thread ("thanks!") scores about 0.5 on resolved: the question was answered, so
#   it half reads as solved. It stays NONE either way; the middle band is Jev saying "unsettled".

## Part 3: Two stages, team then intent

The credit_cards team still has to work out what the customer wants. CLINC150 labels every
message with one of **150 intents**, 15 per team. We find it in two stages: which **team** (10
options), then which of that team's **15 intents**. One flat Choice over all 150 is the other
design; Lab 3.3 races the two.

### 1. The problem

Stage 1 uses Module 1's team question. When it picks the wrong team, **the right intent is not
among stage 2's options**, and stage 2 picks the best of the wrong 15 with no error.

**A two-stage classifier is only as good as its first stage.**

**Figure: The problem: stage 1 picks the wrong team, so the right intent is not among stage 2's options.**

<!-- DIAGRAM: p3-problem-stage-one -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_02/p3-problem-stage-one.png" width="510" alt="Read top to bottom. STATE: the message i have to cancel my reservation after all; beside it, dashed, the answer key, not sent to Jev: cancel_reservation, team kitchen_and_dining. STAGE 1 with criteria TEAMS, a few words per team: Jev picks travel 0.95, WRONG TEAM. STAGE 2 offers travel&#x27;s 15 intents, cancel_reservation is NOT AN OPTION: Jev picks book_hotel 0.67. YOUR CODE: probability 0.95 x 0.67 = 0.64. OUTCOME: book_hotel, wrong with no error, the right intent was never an option; teams right of 40: 29, 11 wrong for good">

### 2. The solution

Describe each team by **what it contains**: its own 15 intents, as data. The lists come from
labelled data (the dataset's `domains.json`); in your own pipeline, group your labels under the
team that handles them.

**The rule: describe options by what they contain, and give each stage what the next one needs.**

**Figure: Criteria as data: the same team question, with each team described by its 15 intents instead of a few words.**

<!-- DIAGRAM: p3-criteria-as-data -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_02/p3-criteria-as-data.png" width="750" alt="Side by side. BEFORE, criteria TEAMS: travel is flights, hotels, luggage, visas; kitchen_and_dining is recipes, cooking, restaurants; 8 more teams in a few words; 532 tokens per call, 29 of 40 teams right. Same question, new criteria. AFTER, criteria TEAM_INTENTS: travel is book_hotel, book_flight and 13 more; kitchen_and_dining holds cancel_reservation among its 15; 1,234 tokens per call, 39 of 40 teams right; stage 2 offers the same lists">

**Figure: The solution: same bands as the problem; stage 1 now sees the intents stage 2 will offer, and lands on the right team.**

<!-- DIAGRAM: p3-solution-criteria-as-data -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_02/p3-solution-criteria-as-data.png" width="570" alt="Same bands as the problem, with the same dashed answer key, not sent to Jev. STAGE 1 with criteria TEAM_INTENTS, each team&#x27;s 15 intents, kitchen_and_dining&#x27;s list holding cancel_reservation: Jev picks kitchen_and_dining 0.97. STAGE 2 offers its 15, cancel_reservation IS AN OPTION: Jev picks it at 1.00. YOUR CODE: probability 0.97 x 1.00 = 0.97. OUTCOME: cancel_reservation, RIGHT, stage 1 saw what stage 2 offers; teams right of 40: 39, 1 wrong">

**Figure: The product of two probabilities: stage 2 only counts if stage 1 was right, so the product is one number for both stages being right.**

<!-- DIAGRAM: p3-product -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_02/p3-product.png" width="750" alt="Two columns for the same message. criteria TEAMS: stage 1 travel 0.95, stage 2 book_hotel 0.67, 0.95 x 0.67 = 0.64, WRONG intent. criteria TEAM_INTENTS: stage 1 kitchen_and_dining 0.97, stage 2 cancel_reservation 1.00, 0.97 x 1.00 = 0.97, RIGHT intent. Both feed YOUR CODE in Lab 3.2: stage 2 only counts if stage 1 was right, so probability = team x intent, both stages right; then needs_review = probability &lt; REVIEW_BELOW. REVIEW_BELOW is a placeholder today; Module 3 chooses it from data">

### 3. Demo

Stage 1 on the same 40 messages as Module 1: first Module 1's text descriptions, then the intent
lists as criteria, with what each costs in tokens and time. Each cell ends with the figure's
message through both stages. You turn that into a function in the lab.

In [ ]:
# ===========================================================================
# DEMO: the problem: stage 1 with Module 1's text descriptions (NARRATE)
# ===========================================================================
# Why: every wrong team here is an intent stage 2 can never reach.
team_question_text = Choice(instructions="Which team should handle `message`?", criteria=TEAMS)


def both_stages(message: str, team_question: Choice) -> None:
    """One message through both stages: stage 2's options are the 15 intents of the team stage 1
    picked, and code multiplies the two probabilities."""
    team = jev.system_one(state={"message": message}, model=JEV_MODEL,
                          questions={"team": team_question}).answers["team"]
    options = {intent: None for intent in TEAM_INTENTS[team.choice]}
    intent = jev.system_one(state={"message": message}, model=JEV_MODEL, questions={
        "intent": Choice(instructions="What does the customer want in `message`?", criteria=options)}).answers["intent"]
    p_team, p_intent = team.probabilities[team.choice], intent.probabilities[intent.choice]
    print(f"  team {team.choice} {p_team:.2f} -> intent {intent.choice} {p_intent:.2f} "
          f"-> product {p_team * p_intent:.2f}")


text_teams, text_ms, text_tokens = [], [], []
for text, intent, team in sample:
    start = time.perf_counter()
    r = jev.system_one(state={"message": text}, model=JEV_MODEL, questions={"team": team_question_text})
    text_ms.append((time.perf_counter() - start) * 1000)
    text_tokens.append(r.usage.input_tokens)
    text_teams.append(r.answers["team"].choice)

wrong = [(row, picked) for row, picked in zip(sample, text_teams) if picked != row[2]]
print(f"Stage 1, text descriptions: {len(sample) - len(wrong)} of {len(sample)} teams right; "
      f"{statistics.mean(text_tokens):.0f} input tokens, p50 {statistics.median(text_ms):.0f} ms per call.")
print(f"{len(wrong)} messages can never get the right intent. Some of them:")
for (text, intent, team), picked in wrong[:6]:
    print(f"  {text[:42]:42} intent {intent:22} sent to {picked:18} (owner: {team})")

print('\n"i have to cancel my reservation after all" (label cancel_reservation), both stages:')
both_stages("i have to cancel my reservation after all", team_question_text)


In [ ]:
# ===========================================================================
# DEMO: the solution: each team described by its own 15 intents (NARRATE)
# ===========================================================================
# Why: the options' descriptions are the data stage 2 will use. No sentence to tune.
team_question_data = Choice(instructions="Which team should handle `message`?", criteria=TEAM_INTENTS)
print("credit_cards is now described as:", TEAM_INTENTS["credit_cards"][:6], "...\n")

data_teams, data_ms, data_tokens = [], [], []
for text, intent, team in sample:
    start = time.perf_counter()
    r = jev.system_one(state={"message": text}, model=JEV_MODEL, questions={"team": team_question_data})
    data_ms.append((time.perf_counter() - start) * 1000)
    data_tokens.append(r.usage.input_tokens)
    data_teams.append(r.answers["team"].choice)

right_text = sum(p == row[2] for row, p in zip(sample, text_teams))
right_data = sum(p == row[2] for row, p in zip(sample, data_teams))
print(f"Stage 1 teams right: text descriptions {right_text}/{len(sample)}, intent lists {right_data}/{len(sample)}")
print(f"Input tokens per call: text {statistics.mean(text_tokens):.0f}, lists {statistics.mean(data_tokens):.0f} "
      f"(${1000 * statistics.mean(data_tokens) * JEV_PRICE_IN:.3f} per 1,000 messages)")
print(f"p50 latency per call: text {statistics.median(text_ms):.0f} ms, lists {statistics.median(data_ms):.0f} ms")
print("Fixed by the intent lists:")
for row, before, after in zip(sample, text_teams, data_teams):
    if before != row[2] and after == row[2]:
        print(f"  {row[0][:42]:42} {before:18} -> {after}")

print('\n"i have to cancel my reservation after all" (label cancel_reservation), both stages:')
both_stages("i have to cancel my reservation after all", team_question_data)


**Figure: Part 3 results: stage 1 teams right on Module 1's 40 messages.**

<!-- DIAGRAM: p3-demo-results -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_02/p3-demo-results.png" width="700" alt="Bar chart of stage 1 teams right out of 40: text descriptions 29, intent lists 39">

**Figure: Part 3 cost: input tokens per stage 1 call for each version, and what 1,000 messages cost.**

<!-- DIAGRAM: p3-demo-tokens -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_02/p3-demo-tokens.png" width="700" alt="Bar chart of input tokens per stage 1 call: text descriptions 532, intent lists 1,234; $0.022 versus $0.052 per 1,000 messages">

> **Think About It**: CLINC150's teams are the dataset authors' grouping, and some calls are
> arguable (is cancelling a restaurant reservation kitchen_and_dining or travel?). The intent
> lists settle most of those for us, because they say how the authors grouped them. Now the
> question the lab ends with: if the intent lists are that good, why not skip stage 1 and ask
> one Choice over all 150 intents? Write down your guess before Level 3.

### 4. Lab 3: A two-stage intent classifier

This is the module's main lab: a classifier that finds the exact intent among 150, in two
stages, with one number your code can gate on. Module 3 tests whether that number deserves trust.

#### Your Task

1. Write the stage 2 question and set a placeholder review threshold (Level 1)
2. Write `classify_intent()`: team, then intent, then one combined probability (Level 2)
3. Optional: race it against one flat Choice over all 150 intents (Level 3)

#### Level 1: Guided

Fill `intent_instructions` (the stage 2 question, about `message`) and `REVIEW_BELOW` (below
this combined probability, a person checks the answer). The cell builds `intent_question(team)`
with your wording and that team's 15 intents as options, and runs stage 2 on 5 messages with the
right team given, so you see stage 2 on its own. Stuck? The FALLBACK cell fills both.

`REVIEW_BELOW` is a placeholder today. You have no data to choose it from yet, and that is the
point: Module 3 measures how often Jev is right at each probability and replaces your number
with one chosen from data.

#### Level 2: Core

Write the body of `classify_intent(message)`: stage 1 with `team_question_data`, stage 2 with
`intent_question(team)`, then compose in code: `probability` = the team's probability times the
intent's probability, and `needs_review` = `probability < REVIEW_BELOW`. The cell runs 40 new
messages and prints team accuracy, intent accuracy, how many go to review, and the accuracy of
the rest, then how many would go to review at 0.5, 0.7 and 0.9. Return the two calls' input
tokens added up as `input_tokens`, and the cell also prints the cost and latency of the full path.

#### Level 3: Challenge (optional)

A take-home: write `flat_check(messages)`. For each message it runs both designs, timed the same
way: one Choice over all 150 intents (the team looked up in code from the intent with
`intent_to_team`), and the two stages (`team_question_data`, then `intent_question(team)`). It
prints, for each design, intent accuracy, team accuracy, p50 latency (the median, in ms) and mean
input tokens. Then answer in a comment: when would you keep two stages?

#### Expected Output

- Level 1: 5 rows; with the right team given, stage 2 finds 4 or 5 of them. The usual miss is
  arguable: "what's the food can easily get in canada" as `meal_suggestion` instead of
  `restaurant_suggestion`
- Level 2: team and intent right for nearly all 40 (40 of 40 in our run). The lowest combined
  probabilities are about 0.6 ("where were you processed"): at 0.5 none go to review, more at
  0.7 and 0.9. On these messages no answer was wrong, so the counts cannot tell a good
  threshold from a bad one yet: that needs wrong answers to compare against, in Module 3.
  The full path is two calls in a row: about twice the tokens and the p50 of one call
- Level 3: two rows of numbers and your answer in a comment

#### Hints

- Level 1: write one plain question about `message` whose answer is one of the 15 options.
  For the threshold, any value from 0.5 to 0.9 is fine today
- Level 2: `response.answers["team"].probabilities[team]` is the team's probability
- Level 3: `{intent: None for intent in intent_to_team}` gives the 150 options;
  `time.perf_counter()` and `response.usage.input_tokens` for the race

**Figure: Lab 3 map: what you build, and how the three levels connect.**

<!-- DIAGRAM: p3-lab-map -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_02/p3-lab-map.png" width="750" alt="WHAT LEVEL 2 USES: the message, team_question_data given for stage 1, intent_question(team) from Level 1 for stage 2, which alone, with the right team given, got 4 of 5, and REVIEW_BELOW, a Level 1 placeholder. LEVEL 2 classify_intent: call 1 with team_question_data gives the team and its probability, call 2 with intent_question(team) gives the intent and its probability, probability is team times intent, needs_review is probability below REVIEW_BELOW. It returns a dict: team, intent, probability, needs_review and input_tokens of both calls; our run on 40 new messages got 40 teams and 40 intents right, and sent 0, 1 to 2, and 4 to review at 0.5, 0.7 and 0.9. Below, dashed, Level 3 optional: flat_check(), 1 Choice over 150 against the two stages timed the same way: accuracy, p50 ms, tokens, and when to keep two stages">

In [ ]:
# ===========================================================================
# SOLUTION: LAB 3.1 (Guided)
# ===========================================================================
# Approach:
# - The question is anchored to the goal (what the customer wants), and the intent names
#   carry the meaning, so the options need no extra description.
# - 0.5 as a start: below it, Jev gave less than even odds to the pair (team, intent).
# Step 1: the stage 2 question. Its options will be one team's 15 intents.
intent_instructions = "What does the customer want in `message`?"

# Step 2: below this combined probability (team x intent), a person checks the answer.
REVIEW_BELOW = 0.5


def intent_question(team: str) -> Choice:
    """Stage 2: a Choice over the 15 intents of one team."""
    return Choice(instructions=intent_instructions,
                  criteria={intent: None for intent in TEAM_INTENTS[team]})


for text, intent, team in random.Random(3).sample(in_scope, 5):   # the right team, given
    a = jev.system_one(state={"message": text}, model=JEV_MODEL,
                       questions={"intent": intent_question(team)}).answers["intent"]
    print(f"{text[:44]:44} {a.choice:24} {a.probabilities[a.choice]:.2f}  "
          f"{'ok' if a.choice == intent else 'expected ' + intent}")
# Explanation:
# - With the right team given, stage 2 is easy: 15 named options. The hard part of the
#   classifier is stage 1, which is why the demo spent its time there.
# - Common mistake: "Which of the 150 intents ..." in a question whose options are 15. The
#   wording must match the options the question really has.

In [ ]:
# ===========================================================================
# SOLUTION: LAB 3.2 (Core)
# ===========================================================================
# Approach:
# - Two calls, because stage 2's options depend on stage 1's answer (the docs' first reason to
#   split a request). Composing happens in code: the team picks the options, and the two
#   probabilities multiply into one number to gate on.
def classify_intent(message: str) -> dict:
    """Two stages: team with team_question_data, then intent with intent_question(team).

    Returns {"team", "team_probability", "intent", "intent_probability", "probability",
    "needs_review", "input_tokens"}: probability = team_probability * intent_probability,
    needs_review = probability < REVIEW_BELOW, and input_tokens = both calls' usage.input_tokens.
    """
    team_response = jev.system_one(state={"message": message}, model=JEV_MODEL,
                                   questions={"team": team_question_data})
    team_answer = team_response.answers["team"]
    team = team_answer.choice
    intent_response = jev.system_one(state={"message": message}, model=JEV_MODEL,
                                     questions={"intent": intent_question(team)})
    intent_answer = intent_response.answers["intent"]
    team_p = team_answer.probabilities[team]
    intent_p = intent_answer.probabilities[intent_answer.choice]
    probability = team_p * intent_p
    return {"team": team, "team_probability": team_p,
            "intent": intent_answer.choice, "intent_probability": intent_p,
            "probability": probability, "needs_review": probability < REVIEW_BELOW,
            "input_tokens": team_response.usage.input_tokens + intent_response.usage.input_tokens}


lab_sample = random.Random(7).sample(in_scope, 40)
results, call_ms = [], []
for text, intent, team in lab_sample:
    start = time.perf_counter()
    result = classify_intent(text)
    call_ms.append((time.perf_counter() - start) * 1000)
    if result is None:
        print("classify_intent() returned None: write its body above.")
        break
    results.append((text, intent, team, result))

if results:
    n = len(results)
    team_right = sum(r["team"] == team for _, _, team, r in results)
    intent_right = sum(r["intent"] == intent for _, intent, _, r in results)
    auto = [(intent, r) for _, intent, _, r in results if not r["needs_review"]]
    auto_right = sum(r["intent"] == intent for intent, r in auto)
    print(f"Team right {team_right}/{n}, intent right {intent_right}/{n}")
    print(f"Sent to review: {n - len(auto)}; the rest: {auto_right}/{len(auto)} right")
    print("Would go to review at 0.5 / 0.7 / 0.9:",
          " / ".join(str(sum(r["probability"] < t for *_, r in results)) for t in (0.5, 0.7, 0.9)))
    tokens = [r.get("input_tokens") for *_, r in results]
    if all(tokens):
        print(f"Full path, 2 calls in a row: {statistics.mean(tokens):.0f} input tokens, "
              f"${1000 * statistics.mean(tokens) * JEV_PRICE_IN:.3f} per 1,000 messages, "
              f"p50 {statistics.median(call_ms):.0f} ms, p95 {statistics.quantiles(call_ms, n=20)[18]:.0f} ms")
    print("\nLowest combined probabilities:")
    for text, intent, team, r in sorted(results, key=lambda x: x[3]["probability"])[:4]:
        print(f"  {r['probability']:.2f} = {r['team_probability']:.2f} x {r['intent_probability']:.2f}  "
              f"{r['intent']:22} (label {intent}) | {text[:34]}")
# Explanation:
# - Multiplying is the plain way to say "both stages must be right". The cookbook on
#   hierarchical classification uses the same product (as a geometric mean for deep trees).
# - The full path is two calls in a row: about twice the tokens and the time of one call. That
#   is the price of two stages; Level 3 measures it against one flat Choice.
# - Our 0.5 is a guess fitted to nothing. Module 3 checks whether these probabilities match
#   how often Jev is right, and picks the threshold from data.
# - Alternative: when stage 1 is torn between two teams, ask stage 2 for both in one request
#   and keep the best product. On our 100-message check it did not change the accuracy.


In [ ]:
# ===========================================================================
# SOLUTION: LAB 3.3 (Challenge, optional)
# ===========================================================================
# Approach:
# - Same messages, two designs, timed the same way. The team in the flat design is a lookup
#   in code: an exact mapping is never a question for Jev.
# Spec: flat_check(messages) runs two designs on every message, timed the same way:
# - flat: ONE Choice over all 150 intents; the team is looked up in code with intent_to_team
# - two stages: team_question_data, then intent_question(team) (it does not need classify_intent)
# For each design it prints intent accuracy, team accuracy, p50 latency (the median) in ms and
# mean input tokens per message.
def flat_check(messages: list[tuple[str, str, str]]) -> None:
    flat_question = Choice(instructions=intent_instructions,
                           criteria={intent: None for intent in intent_to_team})
    rows = {"flat, 1 call": [], "two stages": []}
    for text, intent, team in messages:
        start = time.perf_counter()
        r = jev.system_one(state={"message": text}, model=JEV_MODEL, questions={"intent": flat_question})
        picked = r.answers["intent"].choice
        rows["flat, 1 call"].append((picked == intent, intent_to_team[picked] == team,
                                     (time.perf_counter() - start) * 1000, r.usage.input_tokens))

        start = time.perf_counter()
        r1 = jev.system_one(state={"message": text}, model=JEV_MODEL, questions={"team": team_question_data})
        r2 = jev.system_one(state={"message": text}, model=JEV_MODEL,
                            questions={"intent": intent_question(r1.answers["team"].choice)})
        rows["two stages"].append((r2.answers["intent"].choice == intent, r1.answers["team"].choice == team,
                                   (time.perf_counter() - start) * 1000,
                                   r1.usage.input_tokens + r2.usage.input_tokens))
    n = len(messages)
    for name, rs in rows.items():
        print(f"{name:13} intent {sum(x[0] for x in rs)}/{n}  team {sum(x[1] for x in rs)}/{n}  "
              f"p50 {statistics.median(x[2] for x in rs):4.0f} ms  "
              f"tokens {statistics.mean(x[3] for x in rs):5.0f}")


flat_check(lab_sample[:20])
# Explanation:
# - On CLINC150 the flat Choice is as accurate, faster (one call) and uses fewer tokens: the
#   150 intent names are short and descriptive. Measure before you add a stage.
# - Keep two stages when: the option list is too long for one question (hundreds or more, and
#   the first options get a small pull); each team owns and changes its own intents; the team
#   alone is enough for most routing; or you need to see which stage made a mistake.

**Figure: Known failure modes - TypeSafe's published weak spots for `jev-1.13`, each with its fix and where this module taught it.**

<!-- DIAGRAM: failure-modes -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_02/failure-modes.png" width="750" alt="Two groups from TypeSafe's jev-1.13 weak spots page. TAUGHT IN THIS MODULE: literal reading, Part 1, one judgment per question, asks_notice criteria name edge cases; math and numbers, Part 1, maths in code, days &lt;= WINDOW_DAYS; date comparison, Part 1, month, day and weekday Choices, trip_date() compares in code; indirection, Part 2, name the field, questions name thread. PUBLISHED ONLY, not measured here: large state full of irrelevant detail, filter in code; adversarial content, explicit criteria and testing; instructions and criteria disagree, align the wording; Choice option order leans to the first, reorder and check; generation, a Choice when bounded, an LLM for free text">

Source: docs.typesafe.ai, Model jaggedness, jev-1.13. We pin `jev-1.14.0`, so the dashed group is TypeSafe's word, not something we measured.

# Summary

## What you can now do

1. **Split a vague question.** "Urgent travel notice within 7 days?" hid three judgments and a
   date. On our 10 messages, literal questions plus `trip_date()` in code got 10 of 10 where one
   question got 6 or 7 however we worded it, and a new policy cost zero Jev calls. Be fair to
   the single question: we wrote the 10 messages and chose the 0.7 line while building both
   versions, so this is a small development set, not a held-out test. Try both on messages you
   write yourself.
2. **Design the state.** Jev can only judge what you send it, by name. The whole thread caught
   risks the last line hid; questions that named a field holding only the last line read the
   rest by half.
3. **Describe options by what they contain.** Giving each team its own 15 intents as criteria
   took stage 1 from 29 to 39 of our 40 messages, and a two-stage classifier is only as good as
   its first stage.

### 🧾 The honest part

Two stages are not free: two calls per message instead of one. On CLINC150 the 150 intent names
are short and clear, and when we raced one flat Choice over all 150 against our two stages while
writing this module, the flat Choice was as accurate in one call. That race is not run in class:
Lab 3.3 lets you measure it yourself. Two stages earn their place when the list is too long for
one question, when each team owns and changes its own intents, or when you want to see which
stage made a mistake. Choose by measuring, not by habit.

### Known weak spots

TypeSafe publishes the known weak spots of its models; the figure above maps each one to its fix. The page was written for `jev-1.13` and we pin `jev-1.14.0`: four of them this module worked through, the other five are published only.

## Looking Ahead

Module 3, Measuring and Trusting Jev. Every message gets a team and an intent, even messages that
fit none: "is there a vaccine for ebola" goes to travel at 0.98 and intent vaccines at 1.00. We
measure accuracy and calibration properly, catch out-of-scope messages, compare Jev with a
zero-shot model, and choose `REVIEW_BELOW` from data instead of a guess.

## Homework

Take one question from your own work that starts with "should" or contains "or". Write the
literal questions it hides, the state a person would need to answer them, and the truth table
that turns the answers into the action. Run it on five examples you invent (no real customer
data).